In [0]:
from pyspark.sql.functions import col, current_timestamp

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import DataFrame

def ingest_raw_to_bronze(
    source_path : str,
    target_table : str,
    checkpoint_path : str,
    file_format : str,
    extra_read_options : dict | None = None
) -> None:
    read_options = {
        "cloudFiles.format" : file_format,
        "cloudFiles.schemaLocation" : f"{checkpoint_path}/schema",
        "cloudFiles.schemaEvolutionMode": "addNewColumns"
    }

    delta_write_options = {
        "tblproperties.delta.autoOptimize.optimizeWrite": "true",
        "tblproperties.delta.autoOptimize.autoCompact": "true",
        "checkpointLocation" : f"{checkpoint_path}/write",
    }

    if extra_read_options:
        read_options.update(extra_read_options)

    df_raw = (spark.readStream
              .format("cloudFiles")
              .options(**read_options)
              .load(source_path))
    
    df_bronze = (df_raw
                 .withColumns({
                     "ingesttime" : current_timestamp(),
                     "file_name" : col("_metadata.file_path")
                 })
    )

    query = (df_bronze.writeStream
                .format("delta")
                .outputMode("append")
                .trigger(availableNow=True)
                .options(**delta_write_options)
                .toTable(target_table)
            )
    query.awaitTermination()